# Radeon-Assistant 启动指南

这是 Radeon-Assistant 的启动 Notebook，包含环境配置和应用启动步骤。

## 第一步：安装依赖

In [ ]:
import subprocess
result = subprocess.run(['pip', 'install', '-r', '../requirements.txt'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## 第二步：检查 GPU 环境

In [ ]:
import subprocess
result = subprocess.run(['rocm-smi', '--showproductname'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## 第三步：下载模型（首次运行）

In [ ]:
import subprocess
result = subprocess.run(['python', '../scripts/download_model.py'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## 第四步：启动 Web 应用

In [ ]:
import subprocess
import threading
import time

def run_streamlit():
    result = subprocess.run(['streamlit', 'run', '../ui/web_app.py', '--server.port', '7860'], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

thread = threading.Thread(target=run_streamlit)
thread.start()

print('\n等待应用启动...')
time.sleep(5)
print('\n应用已启动！请在浏览器中访问 http://localhost:7860')

## 第五步：测试推理引擎

In [ ]:
import yaml
from inference.engine import InferenceEngine, InferenceConfig

with open('../config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

model_config = config.get('model', {})
inference_config = InferenceConfig(
    model_path=model_config.get('path', './models/Qwen2.5-7B-Instruct-Q4_K_M.gguf'),
    n_gpu_layers=model_config.get('n_gpu_layers', -1),
    n_ctx=model_config.get('n_ctx', 8192),
    n_batch=model_config.get('n_batch', 512),
    temperature=model_config.get('temperature', 0.7),
    max_tokens=model_config.get('max_tokens', 4096),
    chat_format='qwen2'
)

engine = InferenceEngine(inference_config)
print("模型加载成功")

response = engine.chat_completion([{'role': 'user', 'content': '你好，介绍一下你自己'}])
print(response)

## 停止应用

In [ ]:
import subprocess
result = subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True, text=True)
print('应用已停止')